<a href="https://colab.research.google.com/github/salvadorVMA/navegador/blob/colab_jax/notebooks/colab_jax_gamma_surface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# JAX DR Bridge Estimator — Gamma Surface Sweep

GPU/TPU/CPU-accelerated Doubly Robust Bridge Estimator for the Navegador gamma-surface.

- **float64** precision (with `jax_enable_x64`)
- **`jnp.linalg.solve`** for Newton steps
- **vmap** over bootstrap iterations for massive parallelism

**Workflow:** Setup → Clone → Run sweep (notebook cells or CLI) → Download results.

The compute code lives in standalone Python files under `scripts/`:
- `scripts/jax_dr_bridge.py` — JAX estimator module (all pure functions)
- `scripts/load_pairs.py` — data loading + sweep task builder
- `scripts/run_sweep.py` — batched sweep runner + results summary

These can be run from the notebook (below) or directly from terminal:
```bash
cd /content/navegador
python scripts/run_sweep.py --dataset wvs_temporal --data-repo /content/navegador_data
```

## Cell 1: Setup — Claude Code CLI + JAX

In [1]:
# --- Claude Code CLI ---
!npm install -g @anthropic-ai/claude-code 2>&1 | tail -3

import os
try:
    from google.colab import userdata
    os.environ['ANTHROPIC_API_KEY'] = userdata.get('ANTHROPIC_API_KEY')
    print('ANTHROPIC_API_KEY set from Colab Secrets')
except Exception:
    print('No API key in Colab Secrets — run !claude to authenticate interactively')

# --- JAX setup ---
os.environ['JAX_ENABLE_X64'] = '1'

import jax
jax.config.update('jax_enable_x64', True)
backend = jax.default_backend()
print(f'Backend: {backend}, Devices: {jax.devices()}')

import jax.numpy as jnp
print(f'Float64: {jnp.array(1.0).dtype}')

npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.12.1
npm notice To update run: npm install -g npm@11.12.1
npm notice
No API key in Colab Secrets — run !claude to authenticate interactively


/usr/local/lib/python3.12/dist-packages/jax/_src/cloud_tpu_init.py:86: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(


Backend: tpu, Devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0)]
Float64: float64


## Cell 2: Clone Repos + Set Dataset

Clones the `navegador` repo (branch `colab_jax`, for code + scripts) and `navegador_data` repo (for bridge CSVs + sweep results).

Set `DATASET` to choose which sweep to run:
- `"los_mex"` — 24 domain CSVs, ~4,100 construct pairs (cross-domain)
- `"wvs"` — 66 country CSVs (Wave 7), ~68,000 pairs (cross-country)
- `"wvs_temporal"` — 72 contexts: 7 Mexico waves (W1-W7) + 66 W7 countries, ~86,000 pairs

Both repos are public — no authentication needed.

In [2]:
import os

DATASET = "wvs_temporal"  # "los_mex" | "wvs" | "wvs_temporal"
REPO_DIR = "/content/navegador"
DATA_REPO_DIR = "/content/navegador_data"

if not os.path.exists(REPO_DIR):
    !git clone --depth 1 --branch colab_jax https://github.com/salvadorVMA/navegador.git {REPO_DIR}
else:
    print(f"navegador already cloned at {REPO_DIR}")

if not os.path.exists(DATA_REPO_DIR):
    !git clone --depth 1 https://github.com/salvadorVMA/navegador_data.git {DATA_REPO_DIR}
else:
    print(f"navegador_data already cloned at {DATA_REPO_DIR}")

os.chdir(REPO_DIR)
print(f"Working dir: {os.getcwd()}")
print(f"CLAUDE.md exists: {os.path.exists('CLAUDE.md')}")

Cloning into '/content/navegador'...
remote: Enumerating objects: 780, done.
remote: Counting objects: 100% (780/780), done.
remote: Compressing objects: 100% (711/711), done.
remote: Total 780 (delta 68), reused 651 (delta 57), pack-reused 0 (from 0)
Receiving objects: 100% (780/780), 13.52 MiB | 18.88 MiB/s, done.
Resolving deltas: 100% (68/68), done.
Cloning into '/content/navegador_data'...
remote: Enumerating objects: 170, done.
remote: Counting objects: 100% (170/170), done.
remote: Compressing objects: 100% (120/120), done.
remote: Total 170 (delta 56), reused 154 (delta 48), pack-reused 0 (from 0)
Receiving objects: 100% (170/170), 26.33 MiB | 9.49 MiB/s, done.
Resolving deltas: 100% (56/56), done.
Working dir: /content/navegador
CLAUDE.md exists: True


## Cell 3: Load Data + Build Pairs

Reads the manifest, loads CSVs, and builds the sweep task list.

Equivalent CLI:
```bash
python scripts/load_pairs.py --dataset wvs_temporal --data-repo /content/navegador_data
```

In [3]:
import sys
sys.path.insert(0, REPO_DIR)

from scripts.load_pairs import load_sweep_tasks

context_dfs, sweep_tasks, config = load_sweep_tasks(DATASET, DATA_REPO_DIR)

Loaded 72 contexts, 107,193 total rows.
  WVS_W1_MEX: 1,837 rows x 20 cols
  WVS_W2_MEX: 1,531 rows x 26 cols
  WVS_W3_MEX: 1,510 rows x 29 cols
  WVS_W4_MEX: 1,535 rows x 30 cols
  WVS_W5_MEX: 1,560 rows x 36 cols
  WVS_W6_MEX: 2,000 rows x 49 cols
  WVS_W7_AND: 1,004 rows x 60 cols
  WVS_W7_ARG: 1,003 rows x 60 cols
  WVS_W7_ARM: 1,223 rows x 60 cols
  WVS_W7_AUS: 1,813 rows x 60 cols
  WVS_W7_BGD: 1,200 rows x 60 cols
  WVS_W7_BOL: 2,067 rows x 60 cols
  WVS_W7_BRA: 1,762 rows x 60 cols
  WVS_W7_CAN: 4,018 rows x 60 cols
  WVS_W7_CHL: 1,000 rows x 60 cols
  WVS_W7_CHN: 3,036 rows x 60 cols
  WVS_W7_COL: 1,520 rows x 60 cols
  WVS_W7_CYP: 1,000 rows x 60 cols
  WVS_W7_CZE: 1,200 rows x 60 cols
  WVS_W7_DEU: 1,528 rows x 60 cols
  WVS_W7_ECU: 1,200 rows x 60 cols
  WVS_W7_EGY: 1,200 rows x 60 cols
  WVS_W7_ETH: 1,230 rows x 60 cols
  WVS_W7_GBR: 2,609 rows x 60 cols
  WVS_W7_GRC: 1,200 rows x 60 cols
  WVS_W7_GTM: 1,229 rows x 60 cols
  WVS_W7_HKG: 2,075 rows x 60 cols
  WVS_W7_IDN: 3

## Cell 4: Run Sweep

Runs all pairs with batched checkpointing. Resume-safe — re-run to continue from last checkpoint.

**CLI alternative** (run from terminal, e.g. on a TPU VM):
```bash
cd /content/navegador
python scripts/run_sweep.py --dataset wvs_temporal --data-repo /content/navegador_data \
    --n-sim 2000 --n-bootstrap 200 --batch-size 200
```

In [4]:
from scripts.run_sweep import run_sweep

results = run_sweep(
    context_dfs, sweep_tasks, config,
    n_sim=2000,
    n_bootstrap=200,
    batch_size=200,
)

Remaining: 85807 / 85807
Pre-processing pair data...
Valid pairs to estimate: 69526, pre-skipped: 16281
  [16300/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16350/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16400/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16450/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16500/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16550/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16600/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16650/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16700/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16750/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16800/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16850/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16900/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [16950/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [17000/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [17050/85807] done=0 sig=0 (0.0%) 0.0/s ETA=6952600s
  [17100/85807] 

KeyboardInterrupt: 

## Cell 5: Results Summary + Download

Prints summary statistics and downloads the results JSON.

CLI equivalent:
```bash
python -c "
import json; from scripts.run_sweep import print_summary
print_summary(json.load(open('/content/jax_dr_sweep_wvs_temporal.json')))
"
```

In [ ]:
from scripts.run_sweep import print_summary

print_summary(results)

# Download results JSON
try:
    from google.colab import files
    output_file = f'/content/jax_dr_sweep_{DATASET}.json'
    files.download(output_file)
except ImportError:
    print("Not in Colab — results saved to disk.")